# 10 — The GPU and the seam

**This chapter completes M1: the vertical slice.** The same IR that rendered
readable Python in ch09 now renders WGSL and runs on a real GPU — captures
become a uniform block, the staging buffer you hexdumped in ch08 becomes the
uniform's bytes, and the thesis holds with a Metal device on the other side.

Step 9 leads with **compute** (the 070 decision): the compute-family
contract v1 says a compute kernel's parameters ARE its thread coordinates —
`f(i, j)` runs once per grid point — and the call passes the launch DOMAIN:
`k(out=(W, H))`. The domain is runtime data riding the leaves channel, so
**changing resolution never recompiles**. Backend policies live in the
backend: WGSL has no f64, so this backend narrows to f32 (in the text AND
the layout) — the differential test below compares within f32 tolerance.

Cells tagged `gpu` execute only where an adapter answers; elsewhere the
harness skips them and the committed outputs (baked on an M3 Ultra, Metal)
remain.

In [1]:
import pdum.dsl  # noqa: F401  — batteries: the demo backends route themselves
from pdum.dsl import jit, no_compile
from pdum.dsl.demo.simple_shader import wgsl
from pdum.dsl.kernel.registry import DEFAULT

print("adapter:", wgsl.device().adapter_info if hasattr(wgsl.device(), "adapter_info") else "(ready)")


def make_disk(cx, cy, r, gain):
    @jit(kind="simple_shader.compute")  # demo-scoped kind, routed to the demo WGSL cell
    def disk(i, j):
        x = i / 32.0 - 1.0
        y = j / 32.0 - 1.0
        dx = x - cx
        dy = y - cy
        return gain * (1.0 if dx * dx + dy * dy < r * r else 0.0)

    return disk


img = make_disk(0.1, -0.1, 0.55, 1.0)(out=(64, 64))
shades = " ░▒▓█"
for j in range(0, 64, 4):
    print("".join(shades[int(img[j * 64 + i] * 4)] for i in range(0, 64, 2)))

adapter: <GPUAdapterInfo dict with vendor='', architecture='', device='Apple M3 Ultra', description='', stackgroup_min_size=4, stackgroup_max_size=128, vendor_id=0, device_id=0, adapter_type='IntegratedGPU', backend_type='Metal'>
                                
                                
                                
               ██████           
            ████████████        
          ████████████████      
          █████████████████     
         ██████████████████     
         ██████████████████     
          ████████████████      
           ██████████████       
              █████████         
                                
                                
                                
                                


That image was computed by a WGSL kernel dispatched over a 64×64 grid on
the GPU — captures (`cx, cy, r, gain`) traveled as a uniform block packed
from the same `PackPlan` machinery as every chapter since ch08.

## One IR, two texts

The SAME body, lowered once to the same core IR, rendered by two backends:

In [2]:
key = next(k for k in DEFAULT.specializations._ready if k[2][:2] == ("demo.simple_shader.wgsl", "compute"))
print(DEFAULT.specializations._ready[key].artifact.__pdum_source__)

// nparams=2 staging=16
struct Env {
  m0: f32,
  m4: f32,
  m8: f32,
  m12: f32
}
@group(0) @binding(0) var<uniform> env: Env;
fn kernel_body(p0: f32, p1: f32) -> f32 {
  let v0: f32 = env.m8;
  let v1: f32 = p0;
  let v2: f32 = 32.0;
  let v3: f32 = v1 / v2;
  let v4: f32 = 1.0;
  let v5: f32 = v3 - v4;
  let v6: f32 = env.m0;
  let v7: f32 = v5 - v6;
  let v8: f32 = v7 * v7;
  let v9: f32 = p1;
  let v10: f32 = 32.0;
  let v11: f32 = v9 / v10;
  let v12: f32 = 1.0;
  let v13: f32 = v11 - v12;
  let v14: f32 = env.m4;
  let v15: f32 = v13 - v14;
  let v16: f32 = v15 * v15;
  let v17: f32 = v8 + v16;
  let v18: f32 = env.m12;
  let v19: f32 = env.m12;
  let v20: f32 = v18 * v19;
  let v21: bool = v17 < v20;
  var v24: f32;
  if (v21) {
    let v22: f32 = 1.0;
    v24 = v22;
  } else {
    let v23: f32 = 0.0;
    v24 = v23;
  }
  let v25: f32 = v0 * v24;
  return v25;
}



In [3]:
def make_cpu_twin(cx, cy, r, gain):
    @jit(kind="device")  # identical body, python backend
    def disk(i, j):
        x = i / 32.0 - 1.0
        y = j / 32.0 - 1.0
        dx = x - cx
        dy = y - cy
        return gain * (1.0 if dx * dx + dy * dy < r * r else 0.0)

    return disk


cpu = make_cpu_twin(0.1, -0.1, 0.55, 1.0)
cpu(0.0, 0.0)
pkey = next(k for k in DEFAULT.specializations._ready if k[2][0] == "demo.simple_shader.python")
print(DEFAULT.specializations._ready[pkey].artifact.__pdum_source__)

import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = _u('<d', staging, 16)[0]
    v1 = _u('<d', staging, 32)[0]
    v2 = 32.0
    v3 = v1 / v2
    v4 = 1.0
    v5 = v3 - v4
    v6 = _u('<d', staging, 0)[0]
    v7 = v5 - v6
    v8 = v7 * v7
    v9 = _u('<d', staging, 40)[0]
    v10 = 32.0
    v11 = v9 / v10
    v12 = 1.0
    v13 = v11 - v12
    v14 = _u('<d', staging, 8)[0]
    v15 = v13 - v14
    v16 = v15 * v15
    v17 = v8 + v16
    v18 = _u('<d', staging, 24)[0]
    v19 = _u('<d', staging, 24)[0]
    v20 = v18 * v19
    v21 = v17 < v20
    if v21:
        v22 = 1.0
        v2

## The uniform layout IS the plan

`plan_from_types` with the WGSL backend's `dest_for` policy: the same f64
captures that packed as 8-byte `<d` slots for Python pack as 4-byte `<f`
slots here — **narrowing is a backend layout policy, not a kernel concern**.
(The vec3 align-16 footgun arrives with vec types in the uniform address
space — ch12's problem, M0's `layout.py` documents it.)

In [4]:
from pdum.dsl.demo.simple_shader.wgsl import COMPUTE
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.pack import plan_from_types
from pdum.dsl.kernel.valuekind import BUILTINS

env = (T.f64, T.f64, T.f64, T.f64)  # cx, cy, r, gain
py_plan = plan_from_types(env, (), BUILTINS)
gpu_plan = COMPUTE.plan(env, (), BUILTINS)
print(f"  {'capture':<10} {'python offset:fmt':<20} wgsl offset:fmt")
for name, a, b in zip(("cx", "cy", "r", "gain"), py_plan.slots, gpu_plan.slots):
    print(f"  {name:<10} {a.dest.offset:>3} : {a.dest.fmt:<12} {b.dest.offset:>3} : {b.dest.fmt}")
print(f"  staging: {py_plan.staging_size} bytes on cpu, {gpu_plan.staging_size} on gpu — same values, two ABIs")

  capture    python offset:fmt    wgsl offset:fmt
  cx           0 : <d             0 : <f
  cy           8 : <d             4 : <f
  r           16 : <d             8 : <f
  gain        24 : <d            12 : <f
  staging: 32 bytes on cpu, 16 on gpu — same values, two ABIs


## The differential gate

M1's claim is a *seam*: any backend behind it computes the same function.
The same body runs per-pixel on the CPU (f64) and per-thread on the GPU
(f32); they must agree to f32 precision:

In [5]:
worst = 0.0
for j in range(64):
    for i in range(64):
        worst = max(worst, abs(img[j * 64 + i] - cpu(float(i), float(j))))
print(f"max |gpu - cpu| over 4096 pixels: {worst:.2e}  (f64 vs f32 seam, branch-exact)")

max |gpu - cpu| over 4096 pixels: 0.00e+00  (f64 vs f32 seam, branch-exact)


## The thesis, on a GPU

The loop the project exists for: every frame rebuilds the closure with new
values — and now also changes RESOLUTION mid-loop — under `no_compile()`.
Values move as uniform bytes; the domain moves as launcher data; the
artifact never recompiles.

In [6]:
import time

make_disk(0.0, 0.0, 0.5, 1.0)(out=(64, 64))  # warm (already warm from above)
spec = DEFAULT.specializations
c0, t0 = spec.compiles, time.perf_counter()
with no_compile():
    for f in range(1, 120):
        frame = make_disk(0.4 * (f % 5) / 5, -0.1, 0.3 + 0.002 * f, 1.0)
        frame(out=(64, 64))
    big = make_disk(0.0, 0.0, 0.7, 1.0)(out=(256, 256))  # resolution: runtime data
dt = (time.perf_counter() - t0) / 120
print(f"new compiles across 119 frames + one resolution change: {spec.compiles - c0}")
print(f"per frame (64x64 dispatch + BLOCKING readback): {dt * 1e3:.2f} ms")
print(f"256x256 output arrived with the same artifact: {len(big)} values")
print()
print("microbench readout: the ~ms is the synchronous READBACK round trip —")
print("a render loop that draws instead of reading back pays only write_buffer")
print("+ encode (µs-scale, R12 cost ranking). Readback is the boundary act.")

new compiles across 119 frames + one resolution change: 0
per frame (64x64 dispatch + BLOCKING readback): 1.81 ms
256x256 output arrived with the same artifact: 65536 values

microbench readout: the ~ms is the synchronous READBACK round trip —
a render loop that draws instead of reading back pays only write_buffer
+ encode (µs-scale, R12 cost ranking). Readback is the boundary act.


## The fragment variant

Same machinery, thinner than you'd think: swap the compute stage for a
fullscreen triangle, dispatch for `draw(3)`, storage-out for a texture.
Params become pixel coordinates; the scalar result broadcasts to grayscale
(colors arrive with tuples at step 10). This is M0's disk, reborn behind
the seam:

In [7]:
def make_shader(cx, cy, r):
    @jit(kind="simple_shader.fragment")
    def shader(x, y):
        u = x / 32.0 - 1.0
        v = y / 32.0 - 1.0
        dx = u - cx
        dy = v - cy
        return 1.0 if dx * dx + dy * dy < r * r else 0.0

    return shader


rows = make_shader(-0.2, 0.15, 0.5)(out=(64, 64))
for j in range(0, 64, 4):
    print("".join("█" if rows[j][i] > 0.5 else "·" for i in range(0, 64, 2)))
with no_compile():
    make_shader(0.3, -0.2, 0.35)(out=(64, 64))  # moved disk: uniform bytes only
print("fragment thesis: the moved disk was a cache hit")

································
································
································
································
································
································
········██████████··············
······██████████████············
·····████████████████···········
·····████████████████···········
·····████████████████···········
······██████████████············
········███████████·············
············███·················
································
································
fragment thesis: the moved disk was a cache hit


## Things to notice

- `@workgroup_size` is baked into the WGSL text — pipeline-creation-time by
  spec — so the workgroup size lives in the artifact key exactly where 040
  §3b's block/grid split said it must. Grid (the domain) strips to the
  leaves channel: the 256×256 dispatch was a cache HIT.
- The Env struct's members are named by byte offset (`env.f8`) — the
  uniform layout is the printable plan, not a convention.
- Routing is role-based, and names are honest about their maturity:
  `kind="simple_shader.compute"` / `"simple_shader.fragment"` are
  DEMO-scoped kinds routed to the demo cells — the plain names `compute`
  and `fragment` stay reserved for the real families (080). `kind="device"`
  (the `@jit` default, owned by stdlib since ch04) has no route at all: it
  falls to the registry's DEFAULT backend, which today is the Python demo
  twin. The kinds registered themselves FROM their owning package — the
  ch04 rule, honored by its second customer.

## What we can't do yet

The `[grid, block, smem, stream]` bracket surface (070 §3) — today block is
a fixed backend constant and the domain rides `out=`. Colors and vec types
(tuples, step 10). Arrays and real out-buffers via DLPack (ch12). The
interactive window demo returns with the graphics `draw(target)` surface —
`examples/webgpu/` holds a terminal animation in the meantime. **M1 is
complete: two backends, one IR, the thesis measured on both.**